In [0]:
%python
   import requests
   key = dbutils.secrets.get(scope="finnhub", key="api_key")
   r = requests.get("https://finnhub.io/api/v1/quote", params={"symbol": "AAPL", "token": key})
   print(r.status_code)

In [0]:
%python

import requests
import json
from datetime import datetime, timezone


dbutils.widgets.text("symbol", "MSTR")
symbol = dbutils.widgets.get("symbol").strip().upper()
if not symbol:
    raise ValueError("symbol parameter is empty")
print(f"Ingesting company news for: {symbol}")

api_key = dbutils.secrets.get(scope="finnhub", key="api_key")
ingested_at = datetime.now(timezone.utc).isoformat()


BRONZE_TABLE = "workspace._bronze.finnhub_company_news" 
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace._bronze")

base_url = 'https://finnhub.io/api/v1'

news_url = f'{base_url}/company-news'

news_params = {
    "symbol": symbol,
    "from": "2025-01-01",
    "to": "2026-06-20",
    "token": api_key,
}

news_response = requests.get(news_url, params=news_params)

# Show Finnhub's error message if the call fails (e.g. invalid key)
if news_response.status_code != 200:
    print(news_response.status_code, news_response.text[:200])

news_bronze_record = {
    "source": "finnhub",
    "endpoint": "company-news",
    "symbol": symbol,
    "ingested_at": ingested_at,
    "status_code": news_response.status_code,
    "raw_payload": json.dumps(news_response.json()) if news_response.status_code == 200 else None,
}

df = spark.createDataFrame([news_bronze_record])
 
df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
 
print(f"Status code: {news_bronze_record['status_code']}")
display(df)